# Umwalimu AI — Portfolio Project 2, Part 1: An Intent Classifier
**AITCD001: Chatbots Development**

This notebook adapts the supplied course classifier workflow to Umwalimu AI. It uses the existing six-intent educational dataset, not the fictional bus-ticket example.

## Workflow
1. Audit the CSV and recreate a reproducible stratified 80/20 split (`random_state=42`).
2. Score a transparent provisional keyword baseline on the same test set.
3. Train word and character TF-IDF + Logistic Regression models using training data only.
4. Compare accuracy, precision, recall and F1; show the classification report and confusion matrix.
5. Review confidence thresholds, choose one, compare right/missed/wrong outcomes, and save/reload the selected model.

## Provenance limitations
The available CSV contains 180 manually composed development examples (30 per intent), all marked `source=written`. They are not verified messages collected from 20 real people. Instructor feedback, the original Portfolio Project 1 `UNMATCHED` log, Step 10 local-phrasing examples and the original chatbot rules were not supplied. The rules below are a **provisional dataset-derived baseline**, not the original Umwalimu AI rules. Do not claim those missing evidence requirements are complete.

Run all cells from the repository root.

## Setup — libraries and reproducibility
Install with `pip install pandas scikit-learn joblib matplotlib jupyter`. The fixed random seed makes the split reproducible.

In [ ]:
from pathlib import Path
import re, unicodedata
import joblib, matplotlib.pyplot as plt, numpy as np, pandas as pd, sklearn
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

RANDOM_STATE = 42
DATA_PATH = Path('dataset/intents.csv')
OUTPUT_DIR = Path('dataset')
print('scikit-learn:', sklearn.__version__)

## Step 1 — Inspect, audit and split the dataset
The required fields are `text`, `intent` and `source`. Duplicate detection normalises text only for checking; the original message wording is retained.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(DATA_PATH)
df = pd.read_csv(DATA_PATH)
required = {'text', 'intent', 'source'}
if required - set(df.columns):
    raise ValueError(f'Missing columns: {required - set(df.columns)}')
for col in required:
    if df[col].isna().any():
        raise ValueError(f'Missing values in {col}')
    df[col] = df[col].astype(str).str.strip()
if (df.text == '').any() or (df.intent == '').any():
    raise ValueError('Empty text or intent found')
def norm(s):
    s = unicodedata.normalize('NFKC', str(s)).casefold()
    s = re.sub(r'[^\w\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()
duplicates = int(df.text.map(norm).duplicated().sum())
if duplicates:
    raise ValueError(f'{duplicates} duplicate messages found')
labels = sorted(df.intent.unique())
print('Rows:', len(df), '| intents:', len(labels), '| duplicates:', duplicates)
display(df.intent.value_counts().sort_index().to_frame('messages'))
display(df.source.value_counts().to_frame('rows'))
train_idx, test_idx = train_test_split(df.index.to_numpy(), test_size=0.20, random_state=RANDOM_STATE, stratify=df.intent)
df['split'] = 'train'
df.loc[test_idx, 'split'] = 'test'
df.to_csv(DATA_PATH, index=False)
train_df, test_df = df.loc[train_idx].copy(), df.loc[test_idx].copy()
X_train, y_train = train_df.text.tolist(), train_df.intent.tolist()
X_test, y_test = test_df.text.tolist(), test_df.intent.tolist()
split_counts = df.groupby(['intent','split']).size().unstack(fill_value=0)
display(split_counts)
assert len(train_df) == 144 and len(test_df) == 36
assert (split_counts.train == 24).all() and (split_counts.test == 6).all()
print('Saved reproducible stratified split to', DATA_PATH)

## Step 2 — Provisional rules-based baseline
The original Portfolio Project 1 rules were not available. These explicit keyword rules are only a provisional dataset-derived baseline and must not be described as the original chatbot rules.

In [ ]:
RULES = [
 ('check_answer', r'\b(check|mark|correct|verify|grade|is this right|is my answer|my answer|right answer|reba|igisubizo cyanjye)\b'),
 ('practice_request', r'\b(practice|quiz|test me|test|exercises?|worksheet|revision questions?|questions to practice|imyitozo|ibibazo byimyitozo)\b'),
 ('solve_problem', r'\b(solve|calculate|work out|find x|equation|solve this|answer this|how do i solve|kubara|gukemura)\b'),
 ('explain_concept', r'\b(what is|what are|explain|define|definition|meaning|help me understand|understand|nsobanurira|sobanura|bisobanura)\b'),
 ('learning_guidance', r'\b(study plan|how should i study|how to study|revise|revision plan|what should i learn|learning plan|progress|gahunda yo kwiga|kwiga)\b'),
]
def rule_label(message):
    cleaned = re.sub(r'[^\w\s]', ' ', str(message).casefold())
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    for label, pattern in RULES:
        if re.search(pattern, cleaned, flags=re.IGNORECASE):
            return label
    return 'out_of_scope'
def outcomes(actual, predicted):
    return {
      'right': sum(p == y for p,y in zip(predicted,actual)),
      'missed': sum(p == 'out_of_scope' and y != 'out_of_scope' for p,y in zip(predicted,actual)),
      'wrong': sum(p != 'out_of_scope' and p != y for p,y in zip(predicted,actual)),
    }
rule_predictions = [rule_label(x) for x in X_test]
rule_result = outcomes(y_test, rule_predictions)
print('Provisional rules:', rule_result, '| accuracy:', f'{accuracy_score(y_test,rule_predictions):.1%}')

## Step 3 — Train word and character TF-IDF classifiers
Both vectorizers are fitted only on the training partition. The character model uses character-within-word n-grams (2–4), which can capture spelling variation and informal wording.

In [ ]:
candidates = {
 'words': TfidfVectorizer(),
 'characters': TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4), max_features=1400),
}
models, result_rows = {}, []
for name, vectorizer in candidates.items():
    model = make_pipeline(vectorizer, LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    p, r, f1, _ = precision_recall_fscore_support(y_test, pred, labels=labels, zero_division=0)
    models[name] = model
    result_rows.append({'features':name, 'accuracy':accuracy_score(y_test,pred), 'macro_precision':p.mean(), 'macro_recall':r.mean(), 'macro_f1':f1.mean()})
comparison = pd.DataFrame(result_rows).set_index('features')
display(comparison.style.format('{:.3f}'))
best_name = max(comparison.index, key=lambda n: comparison.loc[n,'accuracy'])
classifier = models[best_name]
predicted = classifier.predict(X_test)
print('Selected:', best_name)

## Step 4 — Classification report and confusion matrix
Precision, recall and F1 are shown for every intent. Rows of the confusion matrix are actual labels; columns are predictions.

In [ ]:
print(classification_report(y_test, predicted, labels=labels, zero_division=0, digits=3))
cm = confusion_matrix(y_test, predicted, labels=labels)
display(pd.DataFrame(cm, index=labels, columns=labels))
fig, ax = plt.subplots(figsize=(9,7))
im = ax.imshow(cm, interpolation='nearest')
ax.set_title(f'Umwalimu AI intent classifier — {best_name}')
ax.set_xlabel('Predicted intent'); ax.set_ylabel('Actual intent')
ax.set_xticks(range(len(labels)), labels=labels, rotation=45, ha='right')
ax.set_yticks(range(len(labels)), labels=labels)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]): ax.text(j, i, str(cm[i,j]), ha='center', va='center')
fig.colorbar(im, ax=ax, label='Messages'); fig.tight_layout(); plt.show()

## Step 5 — Confidence threshold table
Messages whose top class probability is below the threshold are sent to `out_of_scope` as a fallback. Higher thresholds may reduce confidently wrong routes but increase missed requests.

In [ ]:
probabilities = classifier.predict_proba(X_test)
classes = classifier.classes_
threshold_rows = []
for threshold in [0.00,0.10,0.15,0.20,0.25,0.30,0.35,0.40,0.45,0.50]:
    guesses = []
    for row in probabilities:
        i = int(row.argmax()); label = classes[i]
        if row[i] < threshold: label = 'out_of_scope'
        guesses.append(label)
    counts = outcomes(y_test, guesses)
    threshold_rows.append({'threshold':threshold, **counts, 'fallbacks':guesses.count('out_of_scope'), 'fallback_rate':guesses.count('out_of_scope')/len(y_test)})
threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table.style.format({'threshold':'{:.2f}','fallback_rate':'{:.1%}'}))

## Step 6 — Choose threshold and compare systems
For an educational assistant, a confident wrong intent can produce irrelevant teaching content. A fallback asks for clarification but interrupts the learner. The selected threshold is a provisional compromise for this small test set.

In [ ]:
THRESHOLD = 0.30
WHY = ('At 0.30, confident wrong predictions fall from 4 at threshold 0.20 to 1. Correct predictions fall from 29 to 19 and missed requests rise from 3 to 16; the fallback rate is 58.3%. This conservative choice reduces the risk of routing a student's request to an incorrect teaching intent.')
final_predictions = []
for row in probabilities:
    i = int(row.argmax()); label = classes[i]
    if row[i] < THRESHOLD: label = 'out_of_scope'
    final_predictions.append(label)
model_result = outcomes(y_test, final_predictions)
comparison_at_threshold = pd.DataFrame([
 {'system':'Provisional dataset-derived rules', **rule_result},
 {'system':f'{best_name} classifier at {THRESHOLD:.2f}', **model_result},
]).set_index('system')
print('Threshold:', THRESHOLD, '|', WHY)
display(comparison_at_threshold)
print('Fallback rate:', f'{final_predictions.count("out_of_scope")/len(y_test):.1%}')
print('The rules are provisional, not the original chatbot rules.')

## Step 7 — Save, reload and smoke-test the classifier
The joblib file stores the fitted vectorizer and Logistic Regression pipeline together. The following smoke-test messages are illustrative, not genuine participant records.

In [ ]:
MODEL_PATH = OUTPUT_DIR / 'intent_classifier.joblib'
joblib.dump(classifier, MODEL_PATH, compress=('lzma', 9))
loaded = joblib.load(MODEL_PATH)
smoke_messages = [
 'Could you explain what a fraction means?',
 'Please give me five algebra practice questions.',
 'Can you check whether my answer is correct?',
 'How should I plan my revision this week?',
]
smoke_predictions = loaded.predict(smoke_messages)
smoke_confidence = loaded.predict_proba(smoke_messages).max(axis=1)
display(pd.DataFrame({'illustrative_message':smoke_messages,'predicted_intent':smoke_predictions,'top_probability':smoke_confidence}))
assert len(smoke_predictions) == len(smoke_messages)
assert MODEL_PATH.exists() and MODEL_PATH.stat().st_size > 0
print('Saved and reloaded model:', MODEL_PATH, MODEL_PATH.stat().st_size, 'bytes')

## Step 8 — Results and submission checklist

Required files in this folder: this notebook, `intents.csv`, `intent_classifier.joblib`, and `README.md`.

### Outstanding evidence
- At least 20 genuine messages from real people: **not supplied**; current rows are manually composed development examples.
- Assignment 8 written feedback: **not supplied**; feedback-specific changes cannot be verified.
- Portfolio Project 1 `UNMATCHED` log and Step 10 local-phrasing records: **not supplied**.
- Original Portfolio Project 1 chatbot rules: **not supplied**; this notebook uses a provisional dataset-derived baseline.

The technical workflow is executable against the available dataset. Do not claim the missing evidence requirements are complete.